# Data Sparsity Analysis – Hybrid Movie Recommendation System

## What this notebook does

This notebook is the data sparsity part of the group project. The rating matrix is mostly empty, and I want to see what happens to accuracy when it gets emptier.

1. Measure the sparsity of the rating matrix.
2. Keep 100%, 75%, 50%, 25% and 10% of the training ratings, and retrain at each level.
3. Compare user-based CF, item-based CF, FunkSVD and ALS.
4. Score everything on the same held-out test set.

One thing I want to be clear about: a blank cell in the rating matrix is **not** a rating of zero. It means the user never rated that movie. I never fill missing ratings with zero.

## A note on the numbers

The user count, movie count and rating count all come from the CSV files at run time, not from anything hardcoded here. So if the shared split changes, the notebook still runs.

## 1. Imports

- `pandas` to read and organise the CSVs.
- `numpy` for the numeric work.
- `scipy.sparse` to store the mostly-empty rating matrix.
- `scikit-learn` to find similar users / movies.
- `matplotlib` for the graphs.

In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors

## 2. Load the shared train / test split

These files were prepared by the team. The test file stays separate — it is only used at the final evaluation stage, never to fit or tune.

The training file also has `date_*` columns. Those stay in the CSV but are not used as model inputs. Only `userID`, `movieID` and `rating` feed the models.

In [ ]:
data_folder = Path("../dataset/processed")
output_folder = Path("../outputs")
output_folder.mkdir(parents=True, exist_ok=True)

train_path = data_folder / "train_ratings.csv"
test_path = data_folder / "test_ratings.csv"

if not train_path.exists():
    raise FileNotFoundError(f"Training file not found: {train_path}")
if not test_path.exists():
    raise FileNotFoundError(
        f"Test file not found: {test_path}. Place the team's shared test_ratings.csv "
        "inside dataset/processed before running the full evaluation."
    )

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

required_columns = {"userID", "movieID", "rating"}

for name, frame in [("train", train), ("test", test)]:
    missing_columns = required_columns - set(frame.columns)
    if missing_columns:
        raise ValueError(f"{name} is missing columns: {sorted(missing_columns)}")

    if frame[list(required_columns)].isnull().any().any():
        raise ValueError(f"{name} has missing userID, movieID or rating values.")

    if frame.duplicated(["userID", "movieID"]).any():
        raise ValueError(f"{name} contains duplicate userID/movieID pairs.")

print(f"Training ratings: {len(train):,}")
print(f"Testing ratings: {len(test):,}")
print("Columns:", list(train.columns))
print(f"Training users: {train['userID'].nunique():,}")
print(f"Training movies: {train['movieID'].nunique():,}")
print(f"Rating range: {train['rating'].min():g} to {train['rating'].max():g}")

## 3. Original sparsity of the rating matrix

The matrix has `n_users × n_movies` cells. Only the recorded ratings are real interactions; everything else is unobserved.

- density  = recorded ratings ÷ possible pairs × 100
- sparsity = 100 − density

Train and test are combined here only to count the user / movie universe. Rating values from the test set are not used.

In [ ]:
all_ids = pd.concat(
    [train[["userID", "movieID"]], test[["userID", "movieID"]]],
    ignore_index=True
)

user_ids = np.sort(all_ids["userID"].unique())
movie_ids = np.sort(all_ids["movieID"].unique())

n_users = len(user_ids)
n_movies = len(movie_ids)
total_possible = n_users * n_movies
total_recorded = len(train) + len(test)

density = total_recorded / total_possible * 100
overall_sparsity = 100 - density

print(f"Unique users: {n_users:,}")
print(f"Unique movies: {n_movies:,}")
print(f"Possible user-movie pairs: {total_possible:,}")
print(f"Recorded ratings: {total_recorded:,}")
print(f"Matrix density: {density:.4f}%")
print(f"Matrix sparsity: {overall_sparsity:.4f}%")

## 4. How the ratings are distributed

The first plot shows how often each rating value occurs. The other two show how many ratings each user gave and how many each movie received — this tells us whether activity is spread out or concentrated.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

train["rating"].value_counts().sort_index().plot(
    kind="bar", ax=axes[0], title="Training ratings by score"
)
axes[0].set_xlabel("Rating")
axes[0].set_ylabel("Number of ratings")

ratings_per_user = train.groupby("userID").size()
ratings_per_movie = train.groupby("movieID").size()

axes[1].hist(ratings_per_user, bins=30)
axes[1].set_title("Ratings per user")
axes[1].set_xlabel("Number of ratings")
axes[1].set_ylabel("Number of users")

axes[2].hist(ratings_per_movie, bins=30)
axes[2].set_title("Ratings per movie")
axes[2].set_xlabel("Number of ratings")
axes[2].set_ylabel("Number of movies")

plt.tight_layout()
plt.savefig(output_folder / "sparsity_distribution.png", dpi=150)
plt.show()

print("Median ratings per user:", ratings_per_user.median())
print("Median ratings per movie:", ratings_per_movie.median())

## 5. Simulating different levels of available training data

The training rows are shuffled once with a fixed seed. Each smaller sample is taken from the start of the same shuffled list, so:

- 75% contains the 10%, 25% and 50% samples.
- 50% contains the 10% and 25% samples.
- and so on.

The test set never changes.

**Watch out:** 100% / 75% / 50% / 25% / 10% are the share of training ratings kept for an experiment. They are not the fullness of the matrix — even 100% training data is still very sparse.

In [ ]:
availability_levels = [100, 75, 50, 25, 10]
shuffled_train = train.sample(frac=1, random_state=42).reset_index(drop=True)

sparsity_rows = []
for percent in availability_levels:
    sample_size = int(len(shuffled_train) * percent / 100)
    sample = shuffled_train.iloc[:sample_size]

    sample_density = len(sample) / total_possible * 100
    sparsity_rows.append({
        "Data Availability (%)": percent,
        "Training Ratings": len(sample),
        "Matrix Density (%)": sample_density,
        "Matrix Sparsity (%)": 100 - sample_density
    })

sparsity_df = pd.DataFrame(sparsity_rows)
display(sparsity_df.round(4))
sparsity_df.to_csv(output_folder / "sparsity_results.csv", index=False)

plt.figure(figsize=(7, 4))
plt.plot(
    sparsity_df["Data Availability (%)"],
    sparsity_df["Matrix Sparsity (%)"],
    marker="o"
)
plt.xlabel("Available training ratings (%)")
plt.ylabel("Unobserved matrix cells (%)")
plt.title("Matrix sparsity at each training-data level")
plt.gca().invert_xaxis()
plt.grid(True)
plt.tight_layout()
plt.savefig(output_folder / "sparsity_vs_availability.png", dpi=150)
plt.show()

## 6. Real IDs to matrix positions

`userID` and `movieID` are identifiers, not row numbers — user 71534 is not in row 71534 of a small matrix. I build a compact lookup so every real user and movie has an integer position. The map is built from train+test IDs only (again, no ratings used).

The map is needed to place observed training ratings in the right cells.

In [ ]:
user_to_index = {user_id: index for index, user_id in enumerate(user_ids)}
movie_to_index = {movie_id: index for index, movie_id in enumerate(movie_ids)}

def to_arrays(frame):
    user_index = frame["userID"].map(user_to_index).to_numpy(dtype=int)
    movie_index = frame["movieID"].map(movie_to_index).to_numpy(dtype=int)
    ratings = frame["rating"].to_numpy(dtype=float)
    return user_index, movie_index, ratings

test_u, test_m, test_r = to_arrays(test)

rating_min = float(train["rating"].min())
rating_max = float(train["rating"].max())

print(f"Matrix shape: {n_users} users × {n_movies} movies")
print(f"Rating scale in training data: {rating_min:g} to {rating_max:g}")

## 7. Metrics and simple baselines

- **RMSE** – square root of mean squared error. Punishes big mistakes more.
- **MAE** – mean absolute error. The typical mistake in stars.

Two baselines to compare against:

1. **Global mean** – always predict the average training rating.
2. **Bias** – average + user's general generosity + movie's general popularity.

A factor model should beat the bias baseline, otherwise it isn't finding anything beyond simple user and movie tendencies.

In [ ]:
def rmse(actual, predicted):
    return float(np.sqrt(np.mean((actual - predicted) ** 2)))

def mae(actual, predicted):
    return float(np.mean(np.abs(actual - predicted)))

def fit_biases(u, m, r, reg_user=15, reg_item=10):
    # Overall average rating.
    global_mean = float(np.mean(r))

    # How many ratings each movie / user has.
    item_count = np.bincount(m, minlength=n_movies)
    user_count = np.bincount(u, minlength=n_users)

    # Movie bias: how far above / below average a movie is rated.
    # Adding reg_item to the count shrinks the bias towards 0 for movies with few ratings.
    item_sum = np.bincount(m, weights=r - global_mean, minlength=n_movies)
    item_bias = item_sum / (item_count + reg_item)

    # User bias: how generous a user is, after removing the movie effect.
    user_residual = r - global_mean - item_bias[m]
    user_sum = np.bincount(u, weights=user_residual, minlength=n_users)
    user_bias = user_sum / (user_count + reg_user)

    return global_mean, user_bias, item_bias

def predict_global(global_mean, number_of_rows):
    return np.full(number_of_rows, global_mean, dtype=float)

def predict_bias(global_mean, user_bias, item_bias, u, m):
    estimates = global_mean + user_bias[u] + item_bias[m]
    return np.clip(estimates, rating_min, rating_max)

## 8. User-based and item-based collaborative filtering

CF uses rating behaviour to find similar users or similar movies.

- **User-based CF:** find users whose rating patterns resemble the target user. For a movie the target user hasn't rated, average what similar users gave it.
- **Item-based CF:** find movies with similar rating patterns. For a target movie, average what the target user rated similar movies.

Similarity is cosine. The matrix stays sparse — missing cells are not zeros. If no useful neighbour rated the movie, the prediction falls back to the training mean. Each entity keeps at most `NEIGHBOURS` nearest neighbours to keep runtime reasonable.

In [ ]:
NEIGHBOURS = 20

def build_rating_matrix(u, m, r):
    # Only observed ratings are stored. An absent cell is not a zero rating.
    return csr_matrix((r, (u, m)), shape=(n_users, n_movies))

def fit_neighbour_model(u, m, r, mode="user", neighbours=NEIGHBOURS):
    rating_matrix = build_rating_matrix(u, m, r)

    # "Entities" are the things we compare: users (rows) or movies (columns).
    # For item-based CF we transpose so each row is one movie.
    if mode == "user":
        entity_matrix = rating_matrix
    elif mode == "item":
        entity_matrix = rating_matrix.T.tocsr()
    else:
        raise ValueError("mode must be 'user' or 'item'")

    # +1 because every entity's nearest neighbour is itself; we drop it later.
    n_neighbours = min(neighbours + 1, entity_matrix.shape[0])

    model = NearestNeighbors(n_neighbors=n_neighbours, metric="cosine", algorithm="brute")
    model.fit(entity_matrix)

    distances, indices = model.kneighbors(entity_matrix)
    similarities = 1 - distances          # cosine similarity = 1 - cosine distance

    fallback = float(entity_matrix.data.mean()) if entity_matrix.nnz else 0.0
    return entity_matrix, indices, similarities, fallback

def predict_neighbourhood(model, query_u, query_m, mode="user"):
    entity_matrix, neighbour_ids, similarities, fallback = model
    predictions = np.empty(len(query_u), dtype=float)

    for row, (user, movie) in enumerate(zip(query_u, query_m)):
        # User-based: look at similar USERS and what they gave THIS movie.
        # Item-based: look at similar MOVIES and what THIS user gave them.
        # Both cases are the same lookup once the matrix rows are the entities.
        if mode == "user":
            entity, target = user, movie
        else:
            entity, target = movie, user

        ids = neighbour_ids[entity]
        sims = similarities[entity]
        keep_neighbour = ids != entity            # remove the entity itself
        ids, sims = ids[keep_neighbour], sims[keep_neighbour]

        neighbour_ratings = entity_matrix[ids, target].toarray().ravel()
        rated = neighbour_ratings > 0              # which neighbours actually rated it
        weights = np.maximum(sims[rated], 0)       # ignore negative similarities

        if rated.any() and weights.sum() > 0:
            # Similarity-weighted average of the neighbours' ratings.
            predictions[row] = np.dot(weights, neighbour_ratings[rated]) / weights.sum()
        else:
            predictions[row] = fallback

    return np.clip(predictions, rating_min, rating_max)

## 9. Matrix factorisation – the idea

The rating matrix `R` has shape `users × movies` and is mostly empty. Matrix factorisation fills the gaps by writing

    R  ≈  P · Qᵀ   (+ global mean + user bias + movie bias)

- `P` has shape `(n_users, K)` – one row of K numbers per user.
- `Q` has shape `(n_movies, K)` – one row of K numbers per movie.
- The predicted rating for user `u` and movie `i` is the dot product `P[u] · Q[i]`. It only works because both rows have the same length `K`, which is why there are two matrices.

**Where do the numbers come from?** Training. Start with small random values, then adjust them to minimise the squared error on the *observed* ratings only:

    minimise  Σ over observed (u, i)  ( R[u,i] − P[u]·Q[i] )²

The K columns are not labelled "action" or "comedy". They are just whatever numbers make `P · Qᵀ` match the known ratings.

**What does K control?** It is the number of hidden features per user and movie.

- Small K: too simple, cannot capture taste (underfitting).
- Large K: too many parameters, memorises noise (overfitting).
- The number of parameters is `(n_users + n_movies) × K`, so it grows linearly with K.

K is chosen by validation in section 12, not guessed.

## 10. FunkSVD

A quick caution about the name "SVD": a direct truncated SVD on the sparse matrix treats the empty cells as zeros during the decomposition. Those zeros are not real ratings, so that approach is a poor fit for a mostly-empty rating matrix.

Here I use the recommender-system SVD-style approach (FunkSVD). It only touches the ratings that actually exist:

1. Predict using global mean + user bias + movie bias + `P[u]·Q[i]`.
2. Compute the error against the actual rating.
3. Nudge the user and movie factors a little in the direction that reduces the error.
4. Repeat over all observed ratings for a few epochs.

This is not the same as calling `scipy.sparse.linalg.svds` on a zero-filled matrix.

In [ ]:
def fit_svd_style(u, m, r, k=20, epochs=10, learning_rate=0.01, reg=0.05, seed=42):
    global_mean, user_bias, item_bias = fit_biases(u, m, r)

    rng = np.random.default_rng(seed)
    P = rng.normal(0, 0.05, size=(n_users, k))
    Q = rng.normal(0, 0.05, size=(n_movies, k))

    order = np.arange(len(r))

    for epoch in range(epochs):
        rng.shuffle(order)
        for index in order:
            user = u[index]
            movie = m[index]
            actual = r[index]

            user_vector = P[user]
            item_vector = Q[movie]

            prediction = (
                global_mean + user_bias[user] + item_bias[movie]
                + np.dot(user_vector, item_vector)
            )
            error = actual - prediction

            # Standard FunkSVD SGD step. Biases come from fit_biases, so
            # only the factor vectors are updated here.
            P[user]  = user_vector + learning_rate * (error * item_vector - reg * user_vector)
            Q[movie] = item_vector + learning_rate * (error * user_vector - reg * item_vector)

    # Users / movies that never appeared in the training sample still have
    # their random init values. Zero them out so they don't inject noise.
    seen_users = np.unique(u)
    seen_movies = np.unique(m)
    P[np.setdiff1d(np.arange(n_users), seen_users)] = 0.0
    Q[np.setdiff1d(np.arange(n_movies), seen_movies)] = 0.0

    return global_mean, user_bias, item_bias, P, Q

def predict_factor_model(model, u, m):
    global_mean, user_bias, item_bias, P, Q = model
    estimates = (
        global_mean + user_bias[u] + item_bias[m]
        + np.sum(P[u] * Q[m], axis=1)
    )
    return np.clip(estimates, rating_min, rating_max)

## 11. Alternating Least Squares (ALS)

ALS learns the same kind of factor matrices `P` and `Q`, but with a different update rule. Instead of many small gradient steps, it solves each row **exactly**:

1. Hold `Q` fixed. For each user, solve for that user's row of `P`.
2. Hold `P` fixed. For each movie, solve for that movie's row of `Q`.
3. Repeat for a few iterations.

For one user, the exact solution comes from regularised least squares. Let `F` be the factor rows of the movies this user rated (taken from `Q`) and `v` the user's ratings (after removing the bias terms):

    (Fᵀ F + λ·n·I) · p = Fᵀ v        →   solve for p

- `n` is the number of ratings this user gave, so only observed ratings are used.
- `λ` (`lam`) is regularisation. It keeps the factor values from growing too large.
- The system is only `K × K`, so `numpy.linalg.solve` handles it quickly.

The same step works for movies if we swap the roles of users and movies. So the code has one helper, `als_update`, that is used for both.

In [ ]:
def als_update(matrix, fixed, lam):
    """Re-solve every row of one factor matrix while the other one stays fixed.

    matrix : sparse (entities x others) matrix of residual ratings, in CSR format
    fixed  : factor matrix of the "others" (held constant)
    Returns the new factor matrix for the entities.
    """
    k = fixed.shape[1]
    new_factors = np.zeros((matrix.shape[0], k))   # rows with no ratings stay zero

    for row in range(matrix.shape[0]):
        # In CSR format, indptr[row] : indptr[row + 1] marks where this
        # row's observed ratings sit inside the `indices` and `data` arrays.
        start, end = matrix.indptr[row], matrix.indptr[row + 1]
        if start == end:
            continue                               # nothing observed for this row

        other_ids = matrix.indices[start:end]      # which movies (or users) were rated
        values = matrix.data[start:end]            # the ratings themselves
        F = fixed[other_ids]                       # their factor rows

        A = F.T @ F + lam * len(values) * np.eye(k)
        b = F.T @ values
        new_factors[row] = np.linalg.solve(A, b)   # solve A p = b

    return new_factors

def fit_als(u, m, r, k=20, lam=0.1, iterations=10, seed=42):
    global_mean, user_bias, item_bias = fit_biases(u, m, r)

    # Factor model only has to explain what the biases could not.
    residual = r - global_mean - user_bias[u] - item_bias[m]

    user_matrix = csr_matrix((residual, (u, m)), shape=(n_users, n_movies))
    movie_matrix = user_matrix.T.tocsr()           # same data, one row per movie

    rng = np.random.default_rng(seed)
    P = rng.normal(0, 0.05, size=(n_users, k))     # random start (P is overwritten in the first step)
    Q = rng.normal(0, 0.05, size=(n_movies, k))    # random start for Q; P is solved from it first

    for _ in range(iterations):
        P = als_update(user_matrix, Q, lam)        # users, with movies fixed
        Q = als_update(movie_matrix, P, lam)       # movies, with users fixed

    return global_mean, user_bias, item_bias, P, Q

## 12. Choosing K and ALS regularisation with validation data

10% of the current training sample is held out. Each candidate model is fit on the other 90%, the RMSE on the 10% is recorded, and the best setting wins. The **test set is not involved** here. If it were, the final number would be meaningless (that is data leakage).

The grid is kept small (`K = 10, 20` and `lam = 0.1`) because this runs once per sparsity level and the SGD loop is pure Python. A bigger grid is easy to try by editing the two lists below.

In [ ]:
K_VALUES = [10, 20]
ALS_LAM_VALUES = [0.1]

TUNE_EPOCHS = 5      # SGD passes / ALS iterations while tuning (kept short)
FINAL_EPOCHS = 10    # SGD passes / ALS iterations for the final models

def tune_factor_models(u, m, r):
    # Split this sample 90% fit / 10% validation.
    rng = np.random.default_rng(123)
    order = rng.permutation(len(r))
    split = int(0.9 * len(r))
    fit_indices, val_indices = order[:split], order[split:]

    if len(val_indices) == 0:
        raise ValueError("Not enough training ratings to create a validation split.")

    fit_u, fit_m, fit_r = u[fit_indices], m[fit_indices], r[fit_indices]
    val_u, val_m, val_r = u[val_indices], m[val_indices], r[val_indices]

    records = []

    for k in K_VALUES:
        svd_model = fit_svd_style(fit_u, fit_m, fit_r, k=k, epochs=TUNE_EPOCHS, seed=42)
        svd_prediction = predict_factor_model(svd_model, val_u, val_m)
        records.append({
            "Model": "SVD-style", "K": k, "Lambda": np.nan,
            "Validation RMSE": rmse(val_r, svd_prediction)
        })

        for lam in ALS_LAM_VALUES:
            als_model = fit_als(fit_u, fit_m, fit_r, k=k, lam=lam, iterations=TUNE_EPOCHS)
            als_prediction = predict_factor_model(als_model, val_u, val_m)
            records.append({
                "Model": "ALS", "K": k, "Lambda": lam,
                "Validation RMSE": rmse(val_r, als_prediction)
            })

    results = pd.DataFrame(records)
    best_svd = results[results["Model"] == "SVD-style"].sort_values("Validation RMSE").iloc[0]
    best_als = results[results["Model"] == "ALS"].sort_values("Validation RMSE").iloc[0]

    return int(best_svd["K"]), int(best_als["K"]), float(best_als["Lambda"]), results

## 13. Comparing all methods at each sparsity level

For each percentage:

1. Take that percentage of the shuffled training data.
2. Tune the factor-model settings using a validation split from that sample.
3. Train the selected models on the full sample.
4. Predict the same untouched test ratings.
5. Record RMSE, MAE and training time.

The user-based and item-based models are trained on the same sample. If a test user or movie has no useful neighbours or no training interactions, the fallback prediction is used. That is the cold-start limitation, handled in the other notebooks.

In [ ]:
def evaluate_level(percent):
    sample_size = int(len(shuffled_train) * percent / 100)
    sample = shuffled_train.iloc[:sample_size]
    u, m, r = to_arrays(sample)

    # Baselines
    global_mean, user_bias, item_bias = fit_biases(u, m, r)
    global_prediction = predict_global(global_mean, len(test_r))
    bias_prediction = predict_bias(global_mean, user_bias, item_bias, test_u, test_m)

    # Neighbourhood models
    start = time.perf_counter()
    user_model = fit_neighbour_model(u, m, r, mode="user")
    user_cf_prediction = predict_neighbourhood(user_model, test_u, test_m, mode="user")
    user_cf_time = time.perf_counter() - start

    start = time.perf_counter()
    item_model = fit_neighbour_model(u, m, r, mode="item")
    item_cf_prediction = predict_neighbourhood(item_model, test_u, test_m, mode="item")
    item_cf_time = time.perf_counter() - start

    # Select factor-model settings using only this sample's validation split.
    best_svd_k, best_als_k, best_als_lam, tuning_results = tune_factor_models(u, m, r)

    start = time.perf_counter()
    svd_model = fit_svd_style(u, m, r, k=best_svd_k, epochs=FINAL_EPOCHS)
    svd_time = time.perf_counter() - start
    svd_prediction = predict_factor_model(svd_model, test_u, test_m)

    start = time.perf_counter()
    als_model = fit_als(u, m, r, k=best_als_k, lam=best_als_lam, iterations=FINAL_EPOCHS)
    als_time = time.perf_counter() - start
    als_prediction = predict_factor_model(als_model, test_u, test_m)

    predictions = {
        "Global Mean": global_prediction,
        "Bias": bias_prediction,
        "User CF": user_cf_prediction,
        "Item CF": item_cf_prediction,
        "SVD-style": svd_prediction,
        "ALS": als_prediction
    }

    # Sanity checks: catch silent bugs such as NaN or out-of-range predictions.
    for name, p in predictions.items():
        assert len(p) == len(test_r), f"{name}: length mismatch"
        assert not np.isnan(p).any(), f"{name}: contains NaN"
        assert p.min() >= rating_min - 1e-9 and p.max() <= rating_max + 1e-9, \
            f"{name}: predictions outside [{rating_min}, {rating_max}]"

    row = {
        "Data Availability (%)": percent,
        "Training Ratings": len(sample),
        # total_possible uses the train+test ID universe, the same definition at every level.
        "Matrix Sparsity (%)": 100 - len(sample) / total_possible * 100,
        "SVD K": best_svd_k,
        "ALS K": best_als_k,
        "ALS Lambda": best_als_lam,
        "User CF time (s)": user_cf_time,
        "Item CF time (s)": item_cf_time,
        "SVD-style time (s)": svd_time,
        "ALS time (s)": als_time
    }

    for name, prediction in predictions.items():
        row[name + " RMSE"] = rmse(test_r, prediction)
        row[name + " MAE"] = mae(test_r, prediction)

    return row, tuning_results

comparison_rows = []
tuning_tables = {}

for percent in availability_levels:
    print(f"Running experiment with {percent}% of training ratings...")
    result, tuning_table = evaluate_level(percent)
    comparison_rows.append(result)
    tuning_tables[percent] = tuning_table

model_results = pd.DataFrame(comparison_rows)
display(model_results.round(4))

## 13. Results

Lower RMSE and MAE are better. Don't assume one method has to win at every level – read the actual numbers.

The table below shows which K and λ validation picked at each level.

In [ ]:
model_results.to_csv(output_folder / "model_comparison.csv", index=False)

# Validation choices at each level
display(model_results[["Data Availability (%)", "SVD K", "ALS K", "ALS Lambda"]])

metric_names = ["Global Mean", "Bias", "User CF", "Item CF", "SVD-style", "ALS"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for name in metric_names:
    axes[0].plot(model_results["Data Availability (%)"],
                 model_results[name + " RMSE"], marker="o", label=name)
    axes[1].plot(model_results["Data Availability (%)"],
                 model_results[name + " MAE"], marker="o", label=name)

axes[0].set_title("RMSE vs training-data availability")
axes[0].set_xlabel("Training data available (%)")
axes[0].set_ylabel("RMSE (lower is better)")
axes[1].set_title("MAE vs training-data availability")
axes[1].set_xlabel("Training data available (%)")
axes[1].set_ylabel("MAE (lower is better)")

for axis in axes:
    axis.invert_xaxis()
    axis.grid(True)
    axis.legend()

plt.tight_layout()
plt.savefig(output_folder / "algorithm_comparison.png", dpi=150)
plt.show()

## 14. Conclusion – fill in after running

Use your real output values. Don't write invented numbers.

1. What fraction of the user–movie matrix is unobserved?
2. How did RMSE and MAE change as training data availability fell?
3. Which method won at 100%? Did the winner change at lower availability?
4. Did CF struggle when users or movies had very few ratings?
5. Which K and ALS lambda did validation pick at each level?
6. Which method took the longest to train?
7. What are the limitations?

### Limitations
- CF needs overlapping rating behaviour. Users or movies with no ratings are hard.
- MF quality depends on how much data is observed and on the hyperparameters.
- FunkSVD trains on observed ratings only. It is a gradient-descent recommender, not a direct truncation of a zero-filled matrix.
- This notebook handles rating sparsity only. Cold start is handled elsewhere.